In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

In [ ]:
df = pd.read_csv('/content/Dataset_Churn.csv', sep=';')
df.head()

,Customer_ID,Umur,Frek_Belanja,Total_Belanja,Hari_Tidak_Aktif,Membership,Voucher,Churn
0,C001,22,2,250000,45,Silver,Tidak,Ya
1,C002,35,8,3200000,5,Gold,Ya,Tidak
2,C003,28,1,150000,60,Silver,Tidak,Ya
3,C004,41,6,2750000,12,Platinum,Ya,Tidak
4,C005,30,3,450000,35,Silver,Tidak,Ya


In [ ]:
le = LabelEncoder()
df['Membership'] = le.fit_transform(df['Membership'])
df['Voucher'] = le.fit_transform(df['Voucher'])
df['Churn'] = le.fit_transform(df['Churn'])

df.head()

,Customer_ID,Umur,Frek_Belanja,Total_Belanja,Hari_Tidak_Aktif,Membership,Voucher,Churn
0,C001,22,2,250000,45,2,0,1
1,C002,35,8,3200000,5,0,1,0
2,C003,28,1,150000,60,2,0,1
3,C004,41,6,2750000,12,1,1,0
4,C005,30,3,450000,35,2,0,1


In [ ]:
x = df.drop(['Customer_ID','Churn'],axis=1)
y = df['Churn']

x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.3,random_state=42)

In [ ]:
model = RandomForestClassifier()
model.fit(x_train,y_train)

RandomForestClassifier()

In [ ]:
churn_prob = model.predict_proba(x_test)[:,1]
print(churn_prob)

[1.   1.   0.99 0.   0.38 0.03]


In [ ]:
recomendations=[]

for prob in churn_prob :
  if prob>0.8 :
    action = "Voucher 25%"
  elif prob>0.6 :
    action = "Promo Rekomendasi Produk"
  else :
    action = "No Action"
  recomendations.append(action) # Moved inside the loop

print(recomendations)

['Voucher 25%', 'Voucher 25%', 'Voucher 25%', 'No Action', 'No Action', 'No Action']


In [ ]:
result = x_test.copy()

result['Churn_Prob'] = churn_prob
result['Recomendations'] = recomendations

result.head()

,Umur,Frek_Belanja,Total_Belanja,Hari_Tidak_Aktif,Membership,Voucher,Churn_Prob,Recomendations
0,22,2,250000,45,2,0,1.00,Voucher 25%
17,25,2,350000,55,2,0,1.00,Voucher 25%
15,31,3,600000,28,2,0,0.99,Voucher 25%
1,35,8,3200000,5,0,1,0.00,No Action
8,24,4,850000,20,0,1,0.38,No Action


In [ ]:
customer_ids =  df.iloc[x_test.index]['Customer_ID']
result['Customer_ID'] = customer_ids.values

result.head()

,Umur,Frek_Belanja,Total_Belanja,Hari_Tidak_Aktif,Membership,Voucher,Churn_Prob,Recomendations,Customer_ID
0,22,2,250000,45,2,0,1.00,Voucher 25%,C001
17,25,2,350000,55,2,0,1.00,Voucher 25%,C018
15,31,3,600000,28,2,0,0.99,Voucher 25%,C016
1,35,8,3200000,5,0,1,0.00,No Action,C002
8,24,4,850000,20,0,1,0.38,No Action,C009


In [ ]:
result = result[['Customer_ID','Churn_Prob','Recomendations']]

print(result)

   Customer_ID  Churn_Prob Recomendations
0         C001        1.00    Voucher 25%
17        C018        1.00    Voucher 25%
15        C016        0.99    Voucher 25%
1         C002        0.00      No Action
8         C009        0.38      No Action
5         C006        0.03      No Action


In [ ]:
result.to_csv("recomendation_output.csv",index=False)